<a href="https://colab.research.google.com/github/vitorvgsvgs/proj_scrum/blob/main/proj_conectatech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install flask

In [ ]:
from flask import Flask, render_template_string

app = Flask(__name__)

# Cursos iniciais da nossa plataforma
cursos = [
    {
        "id": 1,
        "nome": "Informática Básica",
        "descricao": "Aprenda a utilizar o computador, arquivos e internet."
    },
    {
        "id": 2,
        "nome": "Introdução à Programação com Python",
        "descricao": "Conheça os fundamentos da programação."
    },
    {
        "id": 3,
        "nome": "Criação de Sites",
        "descricao": "Aprenda os primeiros passos com HTML e CSS."
    }
]

# Página inicial da plataforma
@app.route("/")
def inicio():
    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-br">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport"
              content="width=device-width, initial-scale=1.0">

        <title>Plataforma EAD Comunitária</title>

        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 0;
                background: #f2f5f9;
                color: #243247;
            }

            header {
                background: #173b68;
                color: white;
                text-align: center;
                padding: 30px 15px;
            }

            main {
                max-width: 900px;
                margin: 30px auto;
                padding: 0 18px;
            }

            .curso {
                background: white;
                padding: 20px;
                margin: 15px 0;
                border-radius: 10px;
                box-shadow: 0 2px 8px #00000012;
            }

            h2 {
                color: #173b68;
            }

            footer {
                text-align: center;
                padding: 20px;
                color: #526174;
            }
        </style>
    </head>

    <body>
        <header>
            <h1>Plataforma EAD Comunitária</h1>
            <p>Educação gratuita e oportunidades através da tecnologia</p>
        </header>

        <main>
            <h2>Nossos cursos gratuitos</h2>

            <p>
                Aprenda no seu ritmo e desenvolva novas habilidades
                para sua vida pessoal e profissional.
            </p>

            {% for curso in cursos %}
                <section class="curso">
                    <h3>{{ curso.nome }}</h3>
                    <p>{{ curso.descricao }}</p>
                    <p><strong>Curso gratuito</strong></p>
                </section>
            {% endfor %}
        </main>

        <footer>
            Projeto de Extensão I — Gestão da Tecnologia da Informação
        </footer>
    </body>
    </html>
    """, cursos=cursos)

# Teste inicial da aplicação
with app.test_client() as cliente:
    resposta = cliente.get("/")

    print("Status da página:", resposta.status_code)
    print("Página inicial encontrada:",
          b"Plataforma EAD Comunitaria" in resposta.data
          or b"Plataforma EAD Comunit\xc3\xa1ria" in resposta.data)
    print("Quantidade de cursos:", len(cursos))

Status da página: 200
Página inicial encontrada: True
Quantidade de cursos: 3


In [ ]:
import sqlite3

# Nome do arquivo do nosso banco de dados
NOME_BANCO = "plataforma_ead.db"

# Abrir a conexão com o banco
conexao = sqlite3.connect(NOME_BANCO)

# Permitir acessar as colunas pelo nome
conexao.row_factory = sqlite3.Row

print("Conexão com o banco de dados realizada!")

Conexão com o banco de dados realizada!


In [ ]:
# Ativar a verificação de chaves estrangeiras
conexao.execute("PRAGMA foreign_keys = ON")

# Tabela de cursos
conexao.execute("""
CREATE TABLE IF NOT EXISTS cursos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL UNIQUE,
    descricao TEXT NOT NULL,
    ativo INTEGER NOT NULL DEFAULT 1
)
""")

# Tabela de aulas
conexao.execute("""
CREATE TABLE IF NOT EXISTS aulas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    curso_id INTEGER NOT NULL,
    titulo TEXT NOT NULL,
    conteudo TEXT NOT NULL,
    ordem INTEGER NOT NULL,
    FOREIGN KEY (curso_id) REFERENCES cursos(id),
    UNIQUE (curso_id, ordem)
)
""")

# Tabela de alunos
conexao.execute("""
CREATE TABLE IF NOT EXISTS alunos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    data_cadastro TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
)
""")

# Salvar as alterações
conexao.commit()

print("Tabelas criadas com sucesso!")

Tabelas criadas com sucesso!


In [ ]:
cursos_iniciais = [
    (
        "Informática Básica",
        "Aprenda a utilizar o computador, arquivos e internet."
    ),
    (
        "Introdução à Programação com Python",
        "Conheça os fundamentos da programação."
    ),
    (
        "Criação de Sites",
        "Aprenda os primeiros passos com HTML e CSS."
    )
]

conexao.executemany("""
    INSERT OR IGNORE INTO cursos (nome, descricao)
    VALUES (?, ?)
""", cursos_iniciais)

conexao.commit()

print("Cursos cadastrados no banco de dados!")

Cursos cadastrados no banco de dados!


In [ ]:
cursor = conexao.execute("""
    SELECT id, nome, descricao, ativo
    FROM cursos
    ORDER BY id
""")

cursos_cadastrados = cursor.fetchall()

for curso in cursos_cadastrados:
    print(f"ID: {curso['id']}")
    print(f"Nome: {curso['nome']}")
    print(f"Descrição: {curso['descricao']}")
    print(f"Ativo: {curso['ativo']}")
    print("-" * 40)

ID: 1
Nome: Informática Básica
Descrição: Aprenda a utilizar o computador, arquivos e internet.
Ativo: 1
----------------------------------------
ID: 2
Nome: Introdução à Programação com Python
Descrição: Conheça os fundamentos da programação.
Ativo: 1
----------------------------------------
ID: 3
Nome: Criação de Sites
Descrição: Aprenda os primeiros passos com HTML e CSS.
Ativo: 1
----------------------------------------


In [ ]:
# Localizar o ID do curso de Informática Básica
resultado = conexao.execute("""
    SELECT id
    FROM cursos
    WHERE nome = ?
""", ("Informática Básica",)).fetchone()

id_informatica = resultado["id"]

aulas_iniciais = [
    (
        id_informatica,
        "Conhecendo o computador",
        "Introdução aos componentes do computador e suas funções.",
        1
    ),
    (
        id_informatica,
        "Arquivos e pastas",
        "Como criar, organizar, copiar e localizar arquivos e pastas.",
        2
    ),
    (
        id_informatica,
        "Navegação segura na internet",
        "Cuidados básicos com links, senhas e informações pessoais.",
        3
    )
]

conexao.executemany("""
    INSERT OR IGNORE INTO aulas
        (curso_id, titulo, conteudo, ordem)
    VALUES (?, ?, ?, ?)
""", aulas_iniciais)

conexao.commit()

print("Aulas de exemplo cadastradas!")

Aulas de exemplo cadastradas!


In [ ]:
consulta = conexao.execute("""
    SELECT
        cursos.nome AS nome_curso,
        aulas.ordem,
        aulas.titulo,
        aulas.conteudo
    FROM aulas
    INNER JOIN cursos
        ON aulas.curso_id = cursos.id
    ORDER BY cursos.id, aulas.ordem
""")

for aula in consulta.fetchall():
    print(f"Curso: {aula['nome_curso']}")
    print(f"Aula {aula['ordem']}: {aula['titulo']}")
    print(f"Conteúdo: {aula['conteudo']}")
    print("-" * 40)

Curso: Informática Básica
Aula 1: Conhecendo o computador
Conteúdo: Introdução aos componentes do computador e suas funções.
----------------------------------------
Curso: Informática Básica
Aula 2: Arquivos e pastas
Conteúdo: Como criar, organizar, copiar e localizar arquivos e pastas.
----------------------------------------
Curso: Informática Básica
Aula 3: Navegação segura na internet
Conteúdo: Cuidados básicos com links, senhas e informações pessoais.
----------------------------------------


In [ ]:
tabelas = conexao.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""").fetchall()

print("Tabelas existentes no banco:")

for tabela in tabelas:
    print("-", tabela["name"])

Tabelas existentes no banco:
- alunos
- aulas
- cursos


In [ ]:
for tabela in ["cursos", "aulas", "alunos"]:
    total = conexao.execute(
        f"SELECT COUNT(*) FROM {tabela}"
    ).fetchone()[0]

    print(f"{tabela}: {total} registro(s)")

cursos: 3 registro(s)
aulas: 3 registro(s)
alunos: 0 registro(s)


In [ ]:
import sqlite3
from flask import Flask, render_template_string

app = Flask(__name__)

NOME_BANCO = "plataforma_ead.db"

def conectar_banco():
    conexao = sqlite3.connect(NOME_BANCO)
    conexao.row_factory = sqlite3.Row
    conexao.execute("PRAGMA foreign_keys = ON")
    return conexao

print("Flask e SQLite preparados para integração!")

Flask e SQLite preparados para integração!


In [ ]:
@app.route("/")
def inicio():
    conexao = conectar_banco()

    try:
        cursos = conexao.execute("""
            SELECT id, nome, descricao
            FROM cursos
            WHERE ativo = 1
            ORDER BY nome
        """).fetchall()
    finally:
        conexao.close()

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-br">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport"
              content="width=device-width, initial-scale=1.0">

        <title>Plataforma EAD Comunitária</title>

        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 0;
                background: #f2f5f9;
                color: #243247;
            }

            header {
                background: #173b68;
                color: white;
                text-align: center;
                padding: 30px 15px;
            }

            main {
                max-width: 900px;
                margin: 30px auto;
                padding: 0 18px;
            }

            .curso {
                background: white;
                padding: 20px;
                margin: 15px 0;
                border-radius: 10px;
                box-shadow: 0 2px 8px #00000012;
            }

            h2 {
                color: #173b68;
            }

            .mensagem {
                background: white;
                padding: 20px;
                border-radius: 10px;
            }

            footer {
                text-align: center;
                padding: 20px;
                color: #526174;
            }
        </style>
    </head>

    <body>
        <header>
            <h1>Plataforma EAD Comunitária</h1>
            <p>Educação gratuita e oportunidades através da tecnologia</p>
        </header>

        <main>
            <h2>Nossos cursos gratuitos</h2>

            <p>
                Escolha um curso e comece a desenvolver
                suas habilidades tecnológicas.
            </p>

            {% if cursos %}
                {% for curso in cursos %}
                    <section class="curso">
                        <h3>{{ curso["nome"] }}</h3>
                        <p>{{ curso["descricao"] }}</p>
                        <p><strong>Curso gratuito</strong></p>
                    </section>
                {% endfor %}
            {% else %}
                <div class="mensagem">
                    Ainda não há cursos disponíveis.
                </div>
            {% endif %}
        </main>

        <footer>
            Projeto de Extensão I — Gestão da Tecnologia da Informação
        </footer>
    </body>
    </html>
    """, cursos=cursos)

print("Página inicial conectada ao banco de dados!")

Página inicial conectada ao banco de dados!


In [ ]:
cursos = [
    {"nome": "Informática Básica"},
    {"nome": "Criação de Sites"}
]

In [ ]:
cursor = conexao.execute("""
    SELECT id, nome, descricao
    FROM cursos
    WHERE ativo = 1
    ORDER BY nome
""")

for linha in cursor.fetchall():
    print(f"ID: {linha['id']} | Nome: {linha['nome']} | Descrição: {linha['descricao']}")

ID: 3 | Nome: Criação de Sites | Descrição: Aprenda os primeiros passos com HTML e CSS.
ID: 1 | Nome: Informática Básica | Descrição: Aprenda a utilizar o computador, arquivos e internet.
ID: 2 | Nome: Introdução à Programação com Python | Descrição: Conheça os fundamentos da programação.


In [ ]:
from IPython.display import HTML, display

with app.test_client() as cliente:
    resposta = cliente.get("/")

    html = resposta.get_data(as_text=True)

    print("Status HTTP:", resposta.status_code)
    print(
        "Informática Básica encontrada:",
        "Informática Básica" in html
    )
    print(
        "Quantidade de cursos exibidos:",
        html.count('class="curso"')
    )

    # Mostrar uma prévia visual da página no Colab
    if resposta.status_code == 200:
        display(HTML(html))

Status HTTP: 200
Informática Básica encontrada: True
Quantidade de cursos exibidos: 3


In [ ]:
conexao = conectar_banco()

try:
    conexao.execute("""
        INSERT OR IGNORE INTO cursos (nome, descricao)
        VALUES (?, ?)
    """, (
        "Segurança Digital",
        "Aprenda noções de proteção de dados, senhas e navegação segura."
    ))

    conexao.commit()
finally:
    conexao.close()

with app.test_client() as cliente:
    resposta = cliente.get("/")
    html = resposta.get_data(as_text=True)

    print("Status HTTP:", resposta.status_code)
    print(
        "Novo curso aparece na página:",
        "Segurança Digital" in html
    )

    display(HTML(html))

Status HTTP: 200
Novo curso aparece na página: True


In [ ]:
import sqlite3

# Nome do arquivo do nosso banco de dados
NOME_BANCO = "plataforma_ead.db"

# Abrir a conexão com o banco
conexao = sqlite3.connect(NOME_BANCO)

# Permitir acessar as colunas pelo nome
conexao.row_factory = sqlite3.Row

print("Conexão com o banco de dados realizada!")

Conexão com o banco de dados realizada!


In [ ]:
# Ativar a verificação de chaves estrangeiras
conexao.execute("PRAGMA foreign_keys = ON")

# Tabela de cursos
conexao.execute("""
CREATE TABLE IF NOT EXISTS cursos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL UNIQUE,
    descricao TEXT NOT NULL,
    ativo INTEGER NOT NULL DEFAULT 1
)
""")

# Tabela de aulas
conexao.execute("""
CREATE TABLE IF NOT EXISTS aulas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    curso_id INTEGER NOT NULL,
    titulo TEXT NOT NULL,
    conteudo TEXT NOT NULL,
    ordem INTEGER NOT NULL,
    FOREIGN KEY (curso_id) REFERENCES cursos(id),
    UNIQUE (curso_id, ordem)
)
""")

# Tabela de alunos
conexao.execute("""
CREATE TABLE IF NOT EXISTS alunos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    data_cadastro TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
)
""")

# Salvar as alterações
conexao.commit()

print("Tabelas criadas com sucesso!")

Tabelas criadas com sucesso!


In [ ]:
cursos_iniciais = [
    (
        "Informática Básica",
        "Aprenda a utilizar o computador, arquivos e internet."
    ),
    (
        "Introdução à Programação com Python",
        "Conheça os fundamentos da programação."
    ),
    (
        "Criação de Sites",
        "Aprenda os primeiros passos com HTML e CSS."
    )
]

conexao.executemany("""
    INSERT OR IGNORE INTO cursos (nome, descricao)
    VALUES (?, ?)
""", cursos_iniciais)

conexao.commit()

print("Cursos cadastrados no banco de dados!")

Cursos cadastrados no banco de dados!


In [ ]:
cursor = conexao.execute("""
    SELECT id, nome, descricao, ativo
    FROM cursos
    ORDER BY id
""")

cursos_cadastrados = cursor.fetchall()

for curso in cursos_cadastrados:
    print(f"ID: {curso['id']}")
    print(f"Nome: {curso['nome']}")
    print(f"Descrição: {curso['descricao']}")
    print(f"Ativo: {curso['ativo']}")
    print("-" * 40)

ID: 1
Nome: Informática Básica
Descrição: Aprenda a utilizar o computador, arquivos e internet.
Ativo: 1
----------------------------------------
ID: 2
Nome: Introdução à Programação com Python
Descrição: Conheça os fundamentos da programação.
Ativo: 1
----------------------------------------
ID: 3
Nome: Criação de Sites
Descrição: Aprenda os primeiros passos com HTML e CSS.
Ativo: 1
----------------------------------------
ID: 4
Nome: Segurança Digital
Descrição: Aprenda noções de proteção de dados, senhas e navegação segura.
Ativo: 1
----------------------------------------


In [ ]:
# Localizar o ID do curso de Informática Básica
resultado = conexao.execute("""
    SELECT id
    FROM cursos
    WHERE nome = ?
""", ("Informática Básica",)).fetchone()

id_informatica = resultado["id"]

aulas_iniciais = [
    (
        id_informatica,
        "Conhecendo o computador",
        "Introdução aos componentes do computador e suas funções.",
        1
    ),
    (
        id_informatica,
        "Arquivos e pastas",
        "Como criar, organizar, copiar e localizar arquivos e pastas.",
        2
    ),
    (
        id_informatica,
        "Navegação segura na internet",
        "Cuidados básicos com links, senhas e informações pessoais.",
        3
    )
]

conexao.executemany("""
    INSERT OR IGNORE INTO aulas
        (curso_id, titulo, conteudo, ordem)
    VALUES (?, ?, ?, ?)
""", aulas_iniciais)

conexao.commit()

print("Aulas de exemplo cadastradas!")

Aulas de exemplo cadastradas!


In [ ]:
consulta = conexao.execute("""
    SELECT
        cursos.nome AS nome_curso,
        aulas.ordem,
        aulas.titulo,
        aulas.conteudo
    FROM aulas
    INNER JOIN cursos
        ON aulas.curso_id = cursos.id
    ORDER BY cursos.id, aulas.ordem
""")

for aula in consulta.fetchall():
    print(f"Curso: {aula['nome_curso']}")
    print(f"Aula {aula['ordem']}: {aula['titulo']}")
    print(f"Conteúdo: {aula['conteudo']}")
    print("-" * 40)

Curso: Informática Básica
Aula 1: Conhecendo o computador
Conteúdo: Introdução aos componentes do computador e suas funções.
----------------------------------------
Curso: Informática Básica
Aula 2: Arquivos e pastas
Conteúdo: Como criar, organizar, copiar e localizar arquivos e pastas.
----------------------------------------
Curso: Informática Básica
Aula 3: Navegação segura na internet
Conteúdo: Cuidados básicos com links, senhas e informações pessoais.
----------------------------------------


In [ ]:
tabelas = conexao.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""").fetchall()

print("Tabelas existentes no banco:")

for tabela in tabelas:
    print("-", tabela["name"])

Tabelas existentes no banco:
- alunos
- aulas
- cursos


In [ ]:
for tabela in ["cursos", "aulas", "alunos"]:
    total = conexao.execute(
        f"SELECT COUNT(*) FROM {tabela}"
    ).fetchone()[0]

    print(f"{tabela}: {total} registro(s)")

cursos: 4 registro(s)
aulas: 3 registro(s)
alunos: 0 registro(s)


In [ ]:
%%html
<!-- Usamos o comando %%html no topo para que o Colab renderize a estrutura abaixo -->
<section class="curso" style="background: white; padding: 20px; border-radius: 10px; border: 1px solid #ddd;">
    <h3>Nome do Curso</h3>
    <p>Descrição detalhada sobre o que será aprendido no curso.</p>
    <p><strong>Curso gratuito</strong></p>
</section>

In [ ]:
%%html
<!-- Comando mágico %%html adicionado no topo para renderizar o código abaixo -->
<section class="curso" style="background: white; padding: 20px; border-radius: 10px; border: 1px solid #ddd; max-width: 400px;">
    <h3>Nome do Curso</h3>
    <p>Descrição do curso para os alunos.</p>
    <p><strong>Curso gratuito</strong></p>

    <a href="#" style="color: #173b68; text-decoration: none; font-weight: bold;">
        Acessar curso
    </a>
</section>

In [ ]:
from flask import Flask, render_template_string

# Recriamos a aplicação para limpar o estado de requisições anteriores
app = Flask(__name__)

# Redefinimos a rota principal
@app.route("/")
def inicio():
    conexao = conectar_banco()
    try:
        cursos = conexao.execute("""
            SELECT id, nome, descricao
            FROM cursos
            WHERE ativo = 1
            ORDER BY nome
        """).fetchall()
    finally:
        conexao.close()
    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-br">
    <head>
        <meta charset="UTF-8">
        <title>Plataforma EAD Comunitária</title>
        <style>
            body { font-family: Arial, sans-serif; background: #f2f5f9; color: #243247; margin: 0; }
            header { background: #173b68; color: white; text-align: center; padding: 30px 15px; }
            main { max-width: 900px; margin: 30px auto; padding: 0 18px; }
            .curso { background: white; padding: 20px; margin: 15px 0; border-radius: 10px; box-shadow: 0 2px 8px #00000012; }
        </style>
    </head>
    <body>
        <header>
            <h1>Plataforma EAD Comunitária</h1>
        </header>
        <main>
            <h2>Nossos cursos gratuitos</h2>
            {% if cursos %}
                {% for curso in cursos %}
                    <section class="curso">
                        <h3>{{ curso["nome"] }}</h3>
                        <p>{{ curso["descricao"] }}</p>
                        <a href="/curso/{{ curso['id'] }}">Acessar curso</a>
                    </section>
                {% endfor %}
            {% endif %}
        </main>
    </body>
    </html>
    """, cursos=cursos)

# Definimos a nova rota de detalhes do curso
@app.route("/curso/<int:curso_id>")
def pagina_curso(curso_id):
    conexao = conectar_banco()
    try:
        curso = conexao.execute("""
            SELECT id, nome, descricao
            FROM cursos
            WHERE id = ? AND ativo = 1
        """, (curso_id,)).fetchone()

        if curso is None:
            return """
                <h1>Curso não encontrado</h1>
                <a href="/">Voltar aos cursos</a>
            """, 404

        aulas = conexao.execute("""
            SELECT id, titulo, conteudo, ordem
            FROM aulas
            WHERE curso_id = ?
            ORDER BY ordem
        """, (curso_id,)).fetchall()
    finally:
        conexao.close()

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-br">
    <head>
        <meta charset="UTF-8">
        <title>{{ curso["nome"] }} - Plataforma EAD</title>
        <style>
            body { font-family: Arial, sans-serif; background: #f2f5f9; color: #243247; margin: 0; }
            header { background: #173b68; color: white; padding: 25px 15px; text-align: center; }
            main { max-width: 850px; margin: 30px auto; padding: 0 18px; }
            .aula { background: white; padding: 20px; margin: 15px 0; border-radius: 10px; }
        </style>
    </head>
    <body>
        <header>
            <h1>{{ curso["nome"] }}</h1>
            <p>{{ curso["descricao"] }}</p>
        </header>
        <main>
            <a href="/">&larr; Voltar aos cursos</a>
            <h2>Conteúdo do curso</h2>
            {% if aulas %}
                {% for aula in aulas %}
                    <section class="aula">
                        <h3>Aula {{ aula["ordem"] }}: {{ aula["titulo"] }}</h3>
                        <p>{{ aula["conteudo"] }}</p>
                    </section>
                {% endfor %}
            {% else %}
                <p>As aulas deste curso ainda estão sendo preparadas.</p>
            {% endif }
        </main>
    </body>
    </html>
    """, curso=curso, aulas=aulas)

print("Aplicação Flask reiniciada e rotas registradas com sucesso!")

Aplicação Flask reiniciada e rotas registradas com sucesso!


In [ ]:
# Buscar um curso cadastrado para usar no teste
conexao = conectar_banco()

try:
    curso_teste = conexao.execute("""
        SELECT id, nome
        FROM cursos
        WHERE ativo = 1
        ORDER BY id
        LIMIT 1
    """).fetchone()
finally:
    conexao.close()

if curso_teste:
    id_teste = curso_teste["id"]

    with app.test_client() as cliente:
        resposta = cliente.get(f"/curso/{id_teste}")

        html = resposta.get_data(as_text=True)

        print("Status HTTP:", resposta.status_code)
        print("Curso encontrado:", curso_teste["nome"])
        print(
            "Título do curso aparece na página:",
            curso_teste["nome"] in html
        )
        print(
            "Conteúdo das aulas encontrado:",
            "Conteúdo do curso" in html
        )

        from IPython.display import HTML, display

        if resposta.status_code == 200:
            display(HTML(html))
else:
    print("Não há cursos ativos cadastrados para testar.")

ERROR:__main__:Exception on /curso/1 [GET]
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/tmp/ipykernel_1696/3084151532.py", line 78, in pagina_curso
    return render_template_string("""
    <!DOCTYPE html>
    ...<31 lines>...
    </html>
    """, curso=curso, aulas=aulas)
  File "/usr/local/lib/python3.13/dist-packages/fla

Status HTTP: 500
Curso encontrado: Informática Básica
Título do curso aparece na página: False
Conteúdo das aulas encontrado: False


In [ ]:
with app.test_client() as cliente:
    resposta = cliente.get("/curso/999999")

    print("Status HTTP:", resposta.status_code)
    print(
        "Mensagem de curso não encontrado:",
        b"Curso n\xc3\xa3o encontrado" in resposta.data
    )

Status HTTP: 404
Mensagem de curso não encontrado: True


In [ ]:
from flask import render_template_string

# Atualizar a página do curso para incluir links das aulas
def pagina_curso_com_links(curso_id):
    conexao = conectar_banco()

    try:
        curso = conexao.execute("""
            SELECT id, nome, descricao
            FROM cursos
            WHERE id = ? AND ativo = 1
        """, (curso_id,)).fetchone()

        if curso is None:
            return "<h1>Curso não encontrado</h1><a href='/'>Voltar</a>", 404

        aulas = conexao.execute("""
            SELECT id, titulo, conteudo, ordem
            FROM aulas
            WHERE curso_id = ?
            ORDER BY ordem
        """, (curso_id,)).fetchall()

    finally:
        conexao.close()

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-br">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>{{ curso["nome"] }} - Plataforma EAD</title>

        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 0;
                background: #f2f5f9;
                color: #243247;
            }
            header {
                background: #173b68;
                color: white;
                padding: 25px 15px;
                text-align: center;
            }
            main {
                max-width: 850px;
                margin: 30px auto;
                padding: 0 18px;
            }
            .aula {
                background: white;
                padding: 20px;
                margin: 15px 0;
                border-radius: 10px;
            }
            .botao {
                display: inline-block;
                background: #173b68;
                color: white;
                padding: 10px 15px;
                text-decoration: none;
                border-radius: 6px;
            }
            .voltar {
                color: #173b68;
            }
        </style>
    </head>

    <body>
        <header>
            <h1>{{ curso["nome"] }}</h1>
            <p>{{ curso["descricao"] }}</p>
        </header>

        <main>
            <a class="voltar" href="/">← Voltar aos cursos</a>
            <h2>Aulas do curso</h2>

            {% if aulas %}
                {% for aula in aulas %}
                    <section class="aula">
                        <h3>
                            Aula {{ aula["ordem"] }}:
                            {{ aula["titulo"] }}
                        </h3>

                        <a class="botao"
                           href="/aula/{{ aula['id'] }}">
                            Estudar esta aula
                        </a>
                    </section>
                {% endfor %}
            {% else %}
                <p>As aulas deste curso ainda estão sendo preparadas.</p>
            {% endif %}
        </main>
    </body>
    </html>
    """, curso=curso, aulas=aulas)


# Substituir a função da rota já registrada, sem registrar outra rota duplicada
app.view_functions["pagina_curso"] = pagina_curso_com_links

In [ ]:
def pagina_aula(aula_id):
    conexao = conectar_banco()

    try:
        aula = conexao.execute("""
            SELECT
                aulas.id,
                aulas.titulo,
                aulas.conteudo,
                aulas.ordem,
                cursos.id AS curso_id,
                cursos.nome AS nome_curso
            FROM aulas
            INNER JOIN cursos
                ON aulas.curso_id = cursos.id
            WHERE aulas.id = ?
              AND cursos.ativo = 1
        """, (aula_id,)).fetchone()

    finally:
        conexao.close()

    if aula is None:
        return """
            <h1>Aula não encontrada</h1>
            <a href="/">Voltar à página inicial</a>
        """, 404

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-br">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">

        <title>{{ aula["titulo"] }} - Plataforma EAD</title>

        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 0;
                background: #f2f5f9;
                color: #243247;
                line-height: 1.7;
            }

            header {
                background: #173b68;
                color: white;
                padding: 25px 15px;
                text-align: center;
            }

            main {
                max-width: 850px;
                margin: 30px auto;
                padding: 25px;
                background: white;
                border-radius: 10px;
            }

            .conteudo {
                white-space: pre-line;
                margin: 25px 0;
            }

            .botao {
                display: inline-block;
                background: #173b68;
                color: white;
                padding: 10px 15px;
                text-decoration: none;
                border-radius: 6px;
                margin-top: 10px;
            }

            .voltar {
                color: #173b68;
            }
        </style>
    </head>

    <body>
        <header>
            <p>Plataforma EAD Comunitária</p>
            <h1>{{ aula["titulo"] }}</h1>
            <p>
                Curso: {{ aula["nome_curso"] }}
                | Aula {{ aula["ordem"] }}
            </p>
        </header>

        <main>
            <a class="voltar" href="/curso/{{ aula['curso_id'] }}">
                ← Voltar às aulas do curso
            </a>

            <h2>Conteúdo da aula</h2>

            <div class="conteudo">{{ aula["conteudo"] }}</div>

            <a class="botao" href="/curso/{{ aula['curso_id'] }}">
                Voltar ao curso
            </a>
        </main>
    </body>
    </html>
    """, aula=aula)

# Para contornar o AssertionError de adicionar rotas dinâmicas após o app ter iniciado,
# nós associamos diretamente a função nas chaves de visualização do Flask se o mapeamento já existir.
# Se não existir, usamos um truque seguro injetando diretamente no url_map para burlar a trava de segurança.
if "pagina_aula" not in app.view_functions:
    from werkzeug.routing import Rule
    app.url_map.add(Rule("/aula/<int:aula_id>", endpoint="pagina_aula"))

app.view_functions["pagina_aula"] = pagina_aula

print("Página individual de aula configurada!")

Página individual de aula configurada!


In [ ]:
conexao = conectar_banco()
try:
    # Executamos a consulta SQL com um ID de teste (ex: 1)
    aula_id_teste = 1
    cursor = conexao.execute("""
        SELECT aulas.id,
               aulas.titulo,
               aulas.conteudo,
               cursos.id AS curso_id,
               cursos.nome AS nome_curso
        FROM aulas
        INNER JOIN cursos
            ON aulas.curso_id = cursos.id
        WHERE aulas.id = ?
          AND cursos.ativo = 1
    """, (aula_id_teste,))

    resultado = cursor.fetchone()
    if resultado:
        print("Aula Encontrada:")
        print("-" * 40)
        print("Curso:", resultado["nome_curso"])
        print("ID da Aula:", resultado["id"])
        print("Título:", resultado["titulo"])
        print("Conteúdo:", resultado["conteudo"])
    else:
        print(f"Nenhuma aula ativa encontrada com o ID {aula_id_teste}.")
finally:
    conexao.close()

Aula Encontrada:
----------------------------------------
Curso: Informática Básica
ID da Aula: 1
Título: Conhecendo o computador
Conteúdo: Introdução aos componentes do computador e suas funções.


In [ ]:
conexao = conectar_banco()
try:
    # Executamos a consulta SQL de forma integrada ao Python
    aula_id_teste = 1
    cursor = conexao.execute("""
        SELECT aulas.id,
               aulas.titulo,
               aulas.conteudo,
               cursos.id AS curso_id,
               cursos.nome AS nome_curso
        FROM aulas
        INNER JOIN cursos
            ON aulas.curso_id = cursos.id
        WHERE aulas.id = ?
          AND cursos.ativo = 1
    """, (aula_id_teste,))

    resultado = cursor.fetchone()
    if resultado:
        print("Aula Encontrada:")
        print("-" * 40)
        print("Curso:", resultado["nome_curso"])
        print("ID da Aula:", resultado["id"])
        print("Título:", resultado["titulo"])
        print("Conteúdo:", resultado["conteudo"])
    else:
        print(f"Nenhuma aula ativa encontrada com o ID {aula_id_teste}.")
finally:
    conexao.close()

Aula Encontrada:
----------------------------------------
Curso: Informática Básica
ID da Aula: 1
Título: Conhecendo o computador
Conteúdo: Introdução aos componentes do computador e suas funções.


In [ ]:
from IPython.display import HTML, display

conexao = conectar_banco()

try:
    aula_teste = conexao.execute("""
        SELECT id, titulo
        FROM aulas
        ORDER BY curso_id, ordem
        LIMIT 1
    """).fetchone()
finally:
    conexao.close()

if aula_teste:
    id_aula = aula_teste["id"]

    with app.test_client() as cliente:
        resposta = cliente.get(f"/aula/{id_aula}")
        html = resposta.get_data(as_text=True)

        print("Status HTTP:", resposta.status_code)
        print("Aula:", aula_teste["titulo"])
        print(
            "Título encontrado:",
            aula_teste["titulo"] in html
        )

        if resposta.status_code == 200:
            display(HTML(html))
else:
    print("Nenhuma aula cadastrada. Cadastre uma aula antes do teste.")

Status HTTP: 200
Aula: Conhecendo o computador
Título encontrado: True


In [ ]:
from IPython.display import HTML, display

conexao = conectar_banco()

try:
    aula_teste = conexao.execute("""
        SELECT id, titulo
        FROM aulas
        ORDER BY curso_id, ordem
        LIMIT 1
    """).fetchone()
finally:
    conexao.close()

if aula_teste:
    id_aula = aula_teste["id"]

    with app.test_client() as cliente:
        resposta = cliente.get(f"/aula/{id_aula}")
        html = resposta.get_data(as_text=True)

        print("Status HTTP:", resposta.status_code)
        print("Aula:", aula_teste["titulo"])
        print(
            "Título encontrado:",
            aula_teste["titulo"] in html
        )

        if resposta.status_code == 200:
            display(HTML(html))
else:
    print("Nenhuma aula cadastrada. Cadastre uma aula antes do teste.")

Status HTTP: 200
Aula: Conhecendo o computador
Título encontrado: True


In [ ]:
with app.test_client() as cliente:
    resposta = cliente.get("/aula/999999")

    print("Status HTTP:", resposta.status_code)
    print(
        "Aula não encontrada:",
        b"Aula n\xc3\xa3o encontrada" in resposta.data
    )

Status HTTP: 404
Aula não encontrada: True


In [ ]:
# Verificar se a rota da aula está registrada
print("Rota /:", "/" in app.view_functions)
print("Rota da página do curso:",
      "pagina_curso" in app.view_functions)
print("Rota da página da aula:",
      "pagina_aula" in app.view_functions)

# Testar a aplicação sem abrir localhost no navegador
with app.test_client() as cliente:
    resposta_inicio = cliente.get("/")

    print("\nPágina inicial:", resposta_inicio.status_code)

Rota /: False
Rota da página do curso: True
Rota da página da aula: True

Página inicial: 200


In [ ]:
from IPython.display import HTML, display

conexao = conectar_banco()

try:
    aula = conexao.execute("""
        SELECT id, titulo
        FROM aulas
        ORDER BY curso_id, ordem
        LIMIT 1
    """).fetchone()
finally:
    conexao.close()

if aula is None:
    print("Não há aulas cadastradas no banco.")
else:
    with app.test_client() as cliente:
        resposta = cliente.get(f"/aula/{aula['id']}")

        print("Código HTTP:", resposta.status_code)
        print("Aula testada:", aula["titulo"])

        if resposta.status_code == 200:
            display(HTML(resposta.get_data(as_text=True)))
        else:
            print(resposta.get_data(as_text=True))

Código HTTP: 200
Aula testada: Conhecendo o computador


In [ ]:
# Verificar se o Flask está funcionando
print("Aplicação Flask:", app)
print("Rotas registradas:")

for regra in app.url_map.iter_rules():
    print(regra, "->", regra.endpoint)

Aplicação Flask: <Flask '__main__'>
Rotas registradas:
/static/<path:filename> -> static
/ -> inicio
/curso/<int:curso_id> -> pagina_curso
/aula/<int:aula_id> -> pagina_aula


In [ ]:
import threading
import time

# Iniciar o Flask em segundo plano
servidor = threading.Thread(
    target=lambda: app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    ),
    daemon=True
)

servidor.start()
time.sleep(2)

print("Servidor Flask iniciado na porta 5000.")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Servidor Flask iniciado na porta 5000.


In [ ]:

from google.colab.output import eval_js

endereco_publico = eval_js(
    "google.colab.kernel.proxyPort(5002)"
)

print("Endereço de acesso à plataforma:")
print(endereco_publico)


Endereço de acesso à plataforma:
https://5002-m-s-kkb-euw4b1-3qrdx6dx8vrc1-b.europe-west4-1.prod.colab.dev


In [ ]:

from google.colab.output import eval_js

endereco_publico = eval_js(
    "google.colab.kernel.proxyPort(5002)"
)

print("Endereço de acesso à plataforma:")
print(endereco_publico)


Endereço de acesso à plataforma:
https://5002-m-s-kkb-euw4b1-3qrdx6dx8vrc1-b.europe-west4-1.prod.colab.dev


In [ ]:

import requests
import traceback

try:
    resposta = requests.get(
        "http://127.0.0.1:5002/",
        timeout=10
    )

    print("Código HTTP:", resposta.status_code)
    print("Resposta do servidor:")
    print(resposta.text[:3000])

except Exception:
    traceback.print_exc()


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
        conn,
    ...<10 lines>...
        **response_kw,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 493, in _make_

In [ ]:

import threading
import requests
import time
from werkzeug.serving import make_server

# Conferir se a aplicação Flask existe
print("Aplicação Flask:", app)

# Criar o servidor na porta 5002
servidor_http = make_server(
    "0.0.0.0",
    5002,
    app,
    threaded=True
)

# Iniciar o servidor em segundo plano
thread_servidor = threading.Thread(
    target=servidor_http.serve_forever,
    daemon=True
)

thread_servidor.start()
time.sleep(2)

# Testar a página inicial
try:
    resposta = requests.get(
        "http://127.0.0.1:5002/",
        timeout=10
    )

    print("Código HTTP:", resposta.status_code)

    if resposta.status_code == 200:
        print("SUCESSO: servidor funcionando!")
    else:
        print("O servidor respondeu, mas a página apresentou um erro.")

except Exception as erro:
    print("Erro ao testar o servidor:", erro)


Aplicação Flask: <Flask '__main__'>


INFO:werkzeug:127.0.0.1 - - [11/Oct/2026 23:27:10] "GET / HTTP/1.1" 200 -


Código HTTP: 200
SUCESSO: servidor funcionando!


In [ ]:

from google.colab.output import eval_js

endereco_publico = eval_js(
    "google.colab.kernel.proxyPort(5002)"
)

print("Acesse a plataforma neste endereço:")
print(endereco_publico)


Acesse a plataforma neste endereço:
https://5002-m-s-kkb-euw4b1-3qrdx6dx8vrc1-b.europe-west4-1.prod.colab.dev


In [ ]:
from flask import request, redirect, url_for, render_template_string
import sqlite3

def cadastro_aluno():
    mensagem = ""
    tipo_mensagem = ""

    if request.method == "POST":
        nome = request.form.get("nome", "").strip()
        email = request.form.get("email", "").strip().lower()

        if not nome or not email:
            mensagem = "Preencha todos os campos."
            tipo_mensagem = "erro"

        elif "@" not in email or "." not in email.split("@")[-1]:
            mensagem = "Digite um e-mail válido."
            tipo_mensagem = "erro"

        else:
            conexao = conectar_banco()

            try:
                conexao.execute(
                    "INSERT INTO alunos (nome, email) VALUES (?, ?)",
                    (nome, email)
                )
                conexao.commit()

                mensagem = "Cadastro realizado com sucesso!"
                tipo_mensagem = "sucesso"

            except sqlite3.IntegrityError:
                mensagem = "Este e-mail já está cadastrado."
                tipo_mensagem = "erro"

            finally:
                conexao.close()

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Cadastro de aluno - Plataforma EAD</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                background: #f1f5f9;
                margin: 0;
                padding: 30px 15px;
            }
            .container {
                max-width: 450px;
                margin: 30px auto;
                background: white;
                padding: 25px;
                border-radius: 12px;
                box-shadow: 0 3px 12px #00000015;
            }
            h1 { color: #1d4ed8; }
            label { display: block; margin-top: 15px; }
            input {
                width: 100%;
                box-sizing: border-box;
                padding: 12px;
                margin-top: 6px;
                border: 1px solid #cbd5e1;
                border-radius: 6px;
            }
            button {
                width: 100%;
                padding: 12px;
                margin-top: 20px;
                background: #1d4ed8;
                color: white;
                border: 0;
                border-radius: 6px;
                cursor: pointer;
            }
            .sucesso { color: #15803d; }
            .erro { color: #b91c1c; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Cadastro de aluno</h1>
            <p>Crie seu cadastro gratuito para estudar na plataforma.</p>

            {% if mensagem %}
                <p class="{{ tipo_mensagem }}">{{ mensagem }}</p>
            {% endif %}

            <form method="POST">
                <label for="nome">Nome completo</label>
                <input id="nome" name="nome" required
                       maxlength="150" autocomplete="name">

                <label for="email">E-mail</label>
                <input id="email" name="email" type="email"
                       required maxlength="254" autocomplete="email">

                <button type="submit">Realizar cadastro</button>
            </form>

            <p><a href="/">Voltar à página inicial</a></p>
        </div>
    </body>
    </html>
    """, mensagem=mensagem, tipo_mensagem=tipo_mensagem)

# Registramos a rota com segurança usando o mapeamento do url_map para contornar o travamento do Flask
if "cadastro_aluno" not in app.view_functions:
    from werkzeug.routing import Rule
    app.url_map.add(Rule("/cadastro", endpoint="cadastro_aluno", methods=["GET", "POST"]))

app.view_functions["cadastro_aluno"] = cadastro_aluno

print("Página de cadastro registrada com sucesso via injeção de rotas!")

Página de cadastro registrada com sucesso via injeção de rotas!


In [ ]:

with app.test_client() as cliente:
    resposta = cliente.get("/cadastro")

    print("Código HTTP:", resposta.status_code)

    if resposta.status_code == 200:
        print("Página de cadastro funcionando!")
    else:
        print(resposta.get_data(as_text=True)[:2000])


Código HTTP: 200
Página de cadastro funcionando!


In [ ]:

import sqlite3

conexao = conectar_banco()

try:
    colunas = conexao.execute(
        "PRAGMA table_info(alunos)"
    ).fetchall()

    nomes_colunas = [coluna["name"] for coluna in colunas]

    if "senha_hash" not in nomes_colunas:
        conexao.execute(
            "ALTER TABLE alunos ADD COLUMN senha_hash TEXT"
        )
        conexao.commit()
        print("Coluna de senha protegida criada!")
    else:
        print("A coluna senha_hash já existe.")

finally:
    conexao.close()


Coluna de senha protegida criada!


In [ ]:

import sqlite3

conexao = conectar_banco()

try:
    colunas = conexao.execute(
        "PRAGMA table_info(alunos)"
    ).fetchall()

    nomes_colunas = [coluna["name"] for coluna in colunas]

    if "senha_hash" not in nomes_colunas:
        conexao.execute(
            "ALTER TABLE alunos ADD COLUMN senha_hash TEXT"
        )
        conexao.commit()
        print("Coluna de senha protegida criada!")
    else:
        print("A coluna senha_hash já existe.")

finally:
    conexao.close()


A coluna senha_hash já existe.


In [ ]:

import sqlite3

conexao = conectar_banco()

try:
    colunas = conexao.execute(
        "PRAGMA table_info(alunos)"
    ).fetchall()

    nomes_colunas = [coluna["name"] for coluna in colunas]

    if "senha_hash" not in nomes_colunas:
        conexao.execute(
            "ALTER TABLE alunos ADD COLUMN senha_hash TEXT"
        )
        conexao.commit()
        print("Coluna de senha protegida criada!")
    else:
        print("A coluna senha_hash já existe.")

finally:
    conexao.close()


A coluna senha_hash já existe.


In [ ]:
import os
import secrets

# Tenta obter a chave da variável de ambiente
chave = os.environ.get("FLASK_SECRET_KEY")

# Se não estiver configurada ou for muito curta, geramos uma de forma automática e segura
if not chave or len(chave) < 32:
    print("Nenhuma chave forte encontrada nas variáveis de ambiente. Gerando uma chave aleatória segura de 32 bytes...")
    chave = secrets.token_hex(32)

app.secret_key = chave
print("Chave de sessão configurada com sucesso!")

Nenhuma chave forte encontrada nas variáveis de ambiente. Gerando uma chave aleatória segura de 32 bytes...
Chave de sessão configurada com sucesso!


In [ ]:

from flask import request, redirect, session, url_for
from werkzeug.security import generate_password_hash, check_password_hash
import sqlite3

def cadastro_aluno():
    mensagem = ""
    tipo_mensagem = ""

    if request.method == "POST":
        nome = request.form.get("nome", "").strip()
        email = request.form.get("email", "").strip().lower()
        senha = request.form.get("senha", "")

        if not nome or not email or not senha:
            mensagem = "Preencha todos os campos."
            tipo_mensagem = "erro"

        elif len(senha) < 8:
            mensagem = "A senha precisa ter pelo menos 8 caracteres."
            tipo_mensagem = "erro"

        else:
            conexao = conectar_banco()

            try:
                senha_hash = generate_password_hash(senha)

                conexao.execute(
                    """
                    INSERT INTO alunos (nome, email, senha_hash)
                    VALUES (?, ?, ?)
                    """,
                    (nome, email, senha_hash)
                )
                conexao.commit()

                mensagem = "Cadastro realizado com sucesso! Você já pode entrar."
                tipo_mensagem = "sucesso"

            except sqlite3.IntegrityError:
                mensagem = "Este e-mail já está cadastrado."
                tipo_mensagem = "erro"

            finally:
                conexao.close()

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Cadastro - Plataforma EAD</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                background: #f1f5f9;
                padding: 25px 15px;
            }
            .container {
                max-width: 450px;
                margin: 30px auto;
                background: white;
                padding: 25px;
                border-radius: 12px;
            }
            h1 { color: #1d4ed8; }
            label { display: block; margin-top: 15px; }
            input {
                width: 100%;
                box-sizing: border-box;
                padding: 12px;
                margin-top: 6px;
                border: 1px solid #cbd5e1;
                border-radius: 6px;
            }
            button {
                width: 100%;
                padding: 12px;
                margin-top: 20px;
                background: #1d4ed8;
                color: white;
                border: 0;
                border-radius: 6px;
                cursor: pointer;
            }
            .sucesso { color: #15803d; }
            .erro { color: #b91c1c; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Criar conta gratuita</h1>
            <p>Cadastre-se para começar a estudar.</p>

            {% if mensagem %}
                <p class="{{ tipo_mensagem }}">{{ mensagem }}</p>
            {% endif %}

            <form method="POST">
                <label for="nome">Nome completo</label>
                <input id="nome" name="nome" required maxlength="150">

                <label for="email">E-mail</label>
                <input id="email" name="email" type="email"
                       required maxlength="254">

                <label for="senha">Senha (mínimo de 8 caracteres)</label>
                <input id="senha" name="senha" type="password"
                       minlength="8" required autocomplete="new-password">

                <button type="submit">Criar conta</button>
            </form>

            <p><a href="/login">Já tenho uma conta</a></p>
            <p><a href="/">Voltar à página inicial</a></p>
        </div>
    </body>
    </html>
    """, mensagem=mensagem, tipo_mensagem=tipo_mensagem)

# Substituir a rota que já estava registrada
app.view_functions["cadastro_aluno"] = cadastro_aluno

print("Cadastro atualizado para aceitar senhas protegidas!")


Cadastro atualizado para aceitar senhas protegidas!


In [ ]:
from flask import request, redirect, session, url_for, render_template_string
from werkzeug.security import check_password_hash

def login_aluno():
    mensagem = ""

    if request.method == "POST":
        email = request.form.get("email", "").strip().lower()
        senha = request.form.get("senha", "")

        conexao = conectar_banco()

        try:
            aluno = conexao.execute(
                """
                SELECT id, nome, email, senha_hash
                FROM alunos
                WHERE email = ?
                """,
                (email,)
            ).fetchone()
        finally:
            conexao.close()

        if (
            aluno
            and aluno["senha_hash"]
            and check_password_hash(aluno["senha_hash"], senha)
        ):
            session.clear()
            session["aluno_id"] = aluno["id"]
            session["aluno_nome"] = aluno["nome"]

            return redirect(url_for("area_aluno"))

        mensagem = "E-mail ou senha incorretos."

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Login - Plataforma EAD</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                background: #f1f5f9;
                padding: 25px 15px;
            }
            .container {
                max-width: 420px;
                margin: 30px auto;
                padding: 25px;
                background: white;
                border-radius: 12px;
            }
            label { display: block; margin-top: 15px; }
            input {
                width: 100%;
                box-sizing: border-box;
                padding: 12px;
                margin-top: 6px;
                border: 1px solid #cbd5e1;
                border-radius: 6px;
            }
            button {
                width: 100%;
                padding: 12px;
                margin-top: 20px;
                background: #1d4ed8;
                color: white;
                border: 0;
                border-radius: 6px;
            }
            .erro { color: #b91c1c; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Entrar na plataforma</h1>

            {% if mensagem %}
                <p class="erro">{{ mensagem }}</p>
            {% endif %}

            <form method="POST">
                <label for="email">E-mail</label>
                <input id="email" name="email" type="email" required>

                <label for="senha">Senha</label>
                <input id="senha" name="senha" type="password"
                       required autocomplete="current-password">

                <button type="submit">Entrar</button>
            </form>

            <p><a href="/cadastro">Criar uma conta</a></p>
            <p><a href="/">Voltar à página inicial</a></p>
        </div>
    </body>
    </html>
    """)

# Evita o AssertionError usando a injeção manual no mapeamento de URLs
if "login_aluno" not in app.view_functions:
    from werkzeug.routing import Rule
    app.url_map.add(
        Rule(
            "/login",
            endpoint="login_aluno",
            methods=["GET", "POST"]
        )
    )

app.view_functions["login_aluno"] = login_aluno

print("Página de login registrada com sucesso contornando as travas do Flask!")

Página de login registrada com sucesso contornando as travas do Flask!


In [ ]:

def login_aluno():
    mensagem = ""

    if request.method == "POST":
        email = request.form.get("email", "").strip().lower()
        senha = request.form.get("senha", "")

        conexao = conectar_banco()

        try:
            aluno = conexao.execute(
                """
                SELECT id, nome, email, senha_hash
                FROM alunos
                WHERE email = ?
                """,
                (email,)
            ).fetchone()
        finally:
            conexao.close()

        if (
            aluno
            and aluno["senha_hash"]
            and check_password_hash(aluno["senha_hash"], senha)
        ):
            session.clear()
            session["aluno_id"] = aluno["id"]
            session["aluno_nome"] = aluno["nome"]

            return redirect(url_for("area_aluno"))

        mensagem = "E-mail ou senha incorretos."

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Login - Plataforma EAD</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                background: #f1f5f9;
                padding: 25px 15px;
            }
            .container {
                max-width: 420px;
                margin: 30px auto;
                padding: 25px;
                background: white;
                border-radius: 12px;
            }
            label { display: block; margin-top: 15px; }
            input {
                width: 100%;
                box-sizing: border-box;
                padding: 12px;
                margin-top: 6px;
                border: 1px solid #cbd5e1;
                border-radius: 6px;
            }
            button {
                width: 100%;
                padding: 12px;
                margin-top: 20px;
                background: #1d4ed8;
                color: white;
                border: 0;
                border-radius: 6px;
            }
            .erro { color: #b91c1c; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Entrar na plataforma</h1>

            {% if mensagem %}
                <p class="erro">{{ mensagem }}</p>
            {% endif %}

            <form method="POST">
                <label for="email">E-mail</label>
                <input id="email" name="email" type="email" required>

                <label for="senha">Senha</label>
                <input id="senha" name="senha" type="password"
                       required autocomplete="current-password">

                <button type="submit">Entrar</button>
            </form>

            <p><a href="/cadastro">Criar uma conta</a></p>
            <p><a href="/">Voltar à página inicial</a></p>
        </div>
    </body>
    </html>
    """)

if "login_aluno" not in app.view_functions:
    app.add_url_rule(
        "/login",
        endpoint="login_aluno",
        view_func=login_aluno,
        methods=["GET", "POST"]
    )
else:
    app.view_functions["login_aluno"] = login_aluno

print("Página de login registrada!")


Página de login registrada!


In [ ]:

def login_aluno():
    mensagem = ""

    if request.method == "POST":
        email = request.form.get("email", "").strip().lower()
        senha = request.form.get("senha", "")

        conexao = conectar_banco()

        try:
            aluno = conexao.execute(
                """
                SELECT id, nome, email, senha_hash
                FROM alunos
                WHERE email = ?
                """,
                (email,)
            ).fetchone()
        finally:
            conexao.close()

        if (
            aluno
            and aluno["senha_hash"]
            and check_password_hash(aluno["senha_hash"], senha)
        ):
            session.clear()
            session["aluno_id"] = aluno["id"]
            session["aluno_nome"] = aluno["nome"]

            return redirect(url_for("area_aluno"))

        mensagem = "E-mail ou senha incorretos."

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Login - Plataforma EAD</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                background: #f1f5f9;
                padding: 25px 15px;
            }
            .container {
                max-width: 420px;
                margin: 30px auto;
                padding: 25px;
                background: white;
                border-radius: 12px;
            }
            label { display: block; margin-top: 15px; }
            input {
                width: 100%;
                box-sizing: border-box;
                padding: 12px;
                margin-top: 6px;
                border: 1px solid #cbd5e1;
                border-radius: 6px;
            }
            button {
                width: 100%;
                padding: 12px;
                margin-top: 20px;
                background: #1d4ed8;
                color: white;
                border: 0;
                border-radius: 6px;
            }
            .erro { color: #b91c1c; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Entrar na plataforma</h1>

            {% if mensagem %}
                <p class="erro">{{ mensagem }}</p>
            {% endif %}

            <form method="POST">
                <label for="email">E-mail</label>
                <input id="email" name="email" type="email" required>

                <label for="senha">Senha</label>
                <input id="senha" name="senha" type="password"
                       required autocomplete="current-password">

                <button type="submit">Entrar</button>
            </form>

            <p><a href="/cadastro">Criar uma conta</a></p>
            <p><a href="/">Voltar à página inicial</a></p>
        </div>
    </body>
    </html>
    """)

if "login_aluno" not in app.view_functions:
    app.add_url_rule(
        "/login",
        endpoint="login_aluno",
        view_func=login_aluno,
        methods=["GET", "POST"]
    )
else:
    app.view_functions["login_aluno"] = login_aluno

print("Página de login registrada!")


Página de login registrada!


In [ ]:
from functools import wraps
from flask import redirect, url_for, render_template_string, session
from werkzeug.routing import Rule

def exigir_login(funcao):
    @wraps(funcao)
    def wrapper(*args, **kwargs):
        if "aluno_id" not in session:
            return redirect(url_for("login_aluno"))
        return funcao(*args, **kwargs)
    return wrapper

def area_aluno():
    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Área do aluno</title>
    </head>
    <body style="font-family:Arial; max-width:700px; margin:40px auto; padding:15px">
        <h1>Bem-vindo(a), {{ nome }}!</h1>
        <p>Você entrou na sua área de aluno da plataforma EAD.</p>
        <p>Em breve, aqui ficará seu acesso aos cursos e ao seu progresso.</p>
        <p><a href="/">Ver cursos</a></p>
        <p><a href="/sair">Sair da conta</a></p>
    </body>
    </html>
    """, nome=session.get("aluno_nome", "aluno"))

# Registro manual e seguro da rota para contornar o AssertionError
if "area_aluno" not in app.view_functions:
    app.url_map.add(Rule("/area-aluno", endpoint="area_aluno"))

app.view_functions["area_aluno"] = area_aluno

print("Área do aluno criada e registrada com sucesso contornando as travas do Flask!")

Área do aluno criada e registrada com sucesso contornando as travas do Flask!


In [ ]:
from flask import redirect, url_for, session
from werkzeug.routing import Rule

def sair_aluno():
    session.clear()
    return redirect(url_for("login_aluno"))

# Registro dinâmico e seguro para evitar o AssertionError
if "sair_aluno" not in app.view_functions:
    app.url_map.add(Rule("/sair", endpoint="sair_aluno"))

app.view_functions["sair_aluno"] = sair_aluno

print("Rota de saída registrada com sucesso!")

Rota de saída registrada com sucesso!


In [ ]:

with app.test_client() as cliente:
    for caminho in ["/cadastro", "/login", "/area-aluno", "/sair"]:
        resposta = cliente.get(caminho)
        print(caminho, "->", resposta.status_code)


/cadastro -> 200
/login -> 200
/area-aluno -> 200
/sair -> 302


In [ ]:

with app.test_client() as cliente:
    for caminho in ["/cadastro", "/login", "/area-aluno", "/sair"]:
        resposta = cliente.get(caminho)
        print(caminho, "->", resposta.status_code)


/cadastro -> 200
/login -> 200
/area-aluno -> 200
/sair -> 302


In [ ]:

with app.test_client() as cliente:
    for caminho in ["/cadastro", "/login", "/area-aluno", "/sair"]:
        resposta = cliente.get(caminho)
        print(caminho, "->", resposta.status_code)


/cadastro -> 200
/login -> 200
/area-aluno -> 200
/sair -> 302


In [ ]:

def area_aluno():
    conexao = conectar_banco()

    try:
        cursos = conexao.execute("""
            SELECT id, nome, descricao
            FROM cursos
            WHERE ativo = 1
            ORDER BY nome
        """).fetchall()
    finally:
        conexao.close()

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Área do aluno - Plataforma EAD</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                background: #f1f5f9;
                margin: 0;
                padding: 25px 15px;
            }
            .container {
                max-width: 900px;
                margin: auto;
            }
            .cabecalho, .curso {
                background: white;
                padding: 20px;
                margin-bottom: 15px;
                border-radius: 10px;
            }
            h1, h2 { color: #1d4ed8; }
            .botao {
                display: inline-block;
                background: #1d4ed8;
                color: white;
                padding: 10px 16px;
                border-radius: 6px;
                text-decoration: none;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <div class="cabecalho">
                <h1>Olá, {{ nome }}!</h1>
                <p>Bem-vindo(a) à sua área de aprendizagem.</p>
                <a class="botao" href="/sair">Sair da conta</a>
            </div>

            <h2>Meus cursos disponíveis</h2>

            {% if cursos %}
                {% for curso in cursos %}
                    <div class="curso">
                        <h3>{{ curso["nome"] }}</h3>
                        <p>{{ curso["descricao"] }}</p>
                        <a class="botao"
                           href="/curso/{{ curso['id'] }}">
                            Acessar curso
                        </a>
                    </div>
                {% endfor %}
            {% else %}
                <p>Não há cursos disponíveis no momento.</p>
            {% endif %}
        </div>
    </body>
    </html>
    """, nome=session.get("aluno_nome", "aluno"), cursos=cursos)

# Atualizar a função existente, preservando sua rota e proteção
area_original = app.view_functions["area_aluno"]
area_com_login = exigir_login(area_aluno)
app.view_functions["area_aluno"] = area_com_login

print("Área do aluno atualizada com a lista de cursos!")


Área do aluno atualizada com a lista de cursos!


In [ ]:

with app.test_client() as cliente:
    resposta = cliente.get("/area-aluno")

    print("Código HTTP:", resposta.status_code)

    if resposta.status_code in (302, 303):
        print("Proteção funcionando: o visitante precisa entrar.")
    elif resposta.status_code == 200:
        print("Área do aluno acessível nesta sessão de teste.")
    else:
        print(resposta.get_data(as_text=True)[:2000])


Código HTTP: 302
Proteção funcionando: o visitante precisa entrar.


In [ ]:

from functools import wraps
from flask import redirect, url_for, session

# Guardar a função atual que exibe os cursos
pagina_curso_original = app.view_functions["pagina_curso"]

# Criar uma versão protegida
pagina_curso_protegida = exigir_login(pagina_curso_original)

# Substituir a função registrada no Flask
app.view_functions["pagina_curso"] = pagina_curso_protegida

print("A página dos cursos agora exige login!")


A página dos cursos agora exige login!


In [ ]:

# Guardar a função que exibe uma aula
pagina_aula_original = app.view_functions["pagina_aula"]

# Aplicar a proteção de login
pagina_aula_protegida = exigir_login(pagina_aula_original)

# Atualizar a rota registrada
app.view_functions["pagina_aula"] = pagina_aula_protegida

print("As páginas das aulas agora exigem login!")


As páginas das aulas agora exigem login!


In [ ]:

with app.test_client() as cliente:
    testes = [
        ("/curso/1", "Página do curso"),
        ("/aula/1", "Página da aula"),
        ("/area-aluno", "Área do aluno")
    ]

    for caminho, descricao in testes:
        resposta = cliente.get(
            caminho,
            follow_redirects=False
        )

        print(
            descricao,
            "| Código HTTP:", resposta.status_code,
            "| Destino:", resposta.headers.get("Location", "-")
        )


Página do curso | Código HTTP: 302 | Destino: /login
Página da aula | Código HTTP: 302 | Destino: /login
Área do aluno | Código HTTP: 302 | Destino: /login


In [ ]:

conexao = conectar_banco()

try:
    conexao.execute("""
        CREATE TABLE IF NOT EXISTS progresso (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            aluno_id INTEGER NOT NULL,
            aula_id INTEGER NOT NULL,
            concluida_em TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (aluno_id) REFERENCES alunos(id),
            FOREIGN KEY (aula_id) REFERENCES aulas(id),
            UNIQUE (aluno_id, aula_id)
        )
    """)

    conexao.commit()
    print("Tabela de progresso criada com sucesso!")

finally:
    conexao.close()


Tabela de progresso criada com sucesso!


In [ ]:
from flask import abort, redirect, url_for, session
from werkzeug.routing import Rule

def concluir_aula(aula_id):
    aluno_id = session["aluno_id"]

    conexao = conectar_banco()

    try:
        aula = conexao.execute("""
            SELECT id
            FROM aulas
            WHERE id = ?
        """, (aula_id,)).fetchone()

        if aula is None:
            abort(404)

        conexao.execute("""
            INSERT OR IGNORE INTO progresso (aluno_id, aula_id)
            VALUES (?, ?)
        """, (aluno_id, aula_id))

        conexao.commit()

    finally:
        conexao.close()

    return redirect(url_for("pagina_aula", aula_id=aula_id))

# Proteger a rota usando o nosso decorador e registrar dinamicamente
concluir_aula_protegida = exigir_login(concluir_aula)

if "concluir_aula" not in app.view_functions:
    app.url_map.add(Rule("/aula/<int:aula_id>/concluir", endpoint="concluir_aula", methods=["POST"]))

app.view_functions["concluir_aula"] = concluir_aula_protegida

print("Rota para concluir aulas registrada com sucesso contornando as travas do Flask!")

Rota para concluir aulas registrada com sucesso contornando as travas do Flask!


In [ ]:
from flask import session, render_template_string
from werkzeug.routing import Rule

def meu_progresso():
    aluno_id = session["aluno_id"]

    conexao = conectar_banco()

    try:
        resultado = conexao.execute("""
            SELECT
                COUNT(aulas.id) AS total_aulas,
                SUM(
                    CASE WHEN progresso.id IS NOT NULL
                    THEN 1 ELSE 0 END
                ) AS aulas_concluidas
            FROM aulas
            INNER JOIN cursos ON aulas.curso_id = cursos.id
            LEFT JOIN progresso
                ON progresso.aula_id = aulas.id
                AND progresso.aluno_id = ?
            WHERE cursos.ativo = 1
        """, (aluno_id,)).fetchone()

    finally:
        conexao.close()

    total = resultado["total_aulas"] or 0
    concluidas = resultado["aulas_concluidas"] or 0
    percentual = round(concluidas / total * 100) if total else 0

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Meu progresso</title>
    </head>
    <body style="font-family:Arial; max-width:700px; margin:40px auto; padding:15px">
        <h1>Meu progresso</h1>
        <p>Olá, {{ nome }}!</p>
        <p>Aulas concluídas: {{ concluidas }} de {{ total }}</p>
        <progress value="{{ percentual }}" max="100"
                  style="width:100%; height:25px"></progress>
        <p>{{ percentual }}% concluído</p>
        <p><a href="/area-aluno">Voltar à área do aluno</a></p>
    </body>
    </html>
    """,
    nome=session.get("aluno_nome", "aluno"),
    total=total,
    concluidas=concluidas,
    percentual=percentual)

# Proteger a função com o login obrigatório
meu_progresso_protegido = exigir_login(meu_progresso)

# Registrar manualmente a rota na tabela do Flask
if "meu_progresso" not in app.view_functions:
    app.url_map.add(Rule("/meu-progresso", endpoint="meu_progresso"))

app.view_functions["meu_progresso"] = meu_progresso_protegido

print("Página de progresso criada e registrada com sucesso contornando as travas do Flask!")

Página de progresso criada e registrada com sucesso contornando as travas do Flask!


In [ ]:

with app.test_client() as cliente:
    resposta = cliente.get(
        "/meu-progresso",
        follow_redirects=False
    )

    print("Código HTTP:", resposta.status_code)
    print("Destino:", resposta.headers.get("Location", "-"))


Código HTTP: 302
Destino: /login


In [ ]:

conexao = conectar_banco()

try:
    conexao.execute("""
        CREATE TABLE IF NOT EXISTS recuperacao_senha (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            aluno_id INTEGER NOT NULL,
            token_hash TEXT NOT NULL UNIQUE,
            expira_em TEXT NOT NULL,
            utilizado INTEGER NOT NULL DEFAULT 0,
            FOREIGN KEY (aluno_id) REFERENCES alunos(id)
        )
    """)
    conexao.commit()
    print("Estrutura de recuperação de senha criada!")
finally:
    conexao.close()


Estrutura de recuperação de senha criada!


In [ ]:
import secrets
import hashlib
from datetime import datetime, timedelta
from flask import request, redirect, url_for, render_template_string, session
from werkzeug.security import generate_password_hash
import sqlite3
from werkzeug.routing import Rule

# Atualizar a página de login com o link de recuperação
def login_aluno():
    mensagem = ""

    if request.method == "POST":
        email = request.form.get("email", "").strip().lower()
        senha = request.form.get("senha", "")

        conexao = conectar_banco()
        try:
            aluno = conexao.execute("""
                SELECT id, nome, email, senha_hash
                FROM alunos
                WHERE email = ?
            """, (email,)).fetchone()
        finally:
            conexao.close()

        from werkzeug.security import check_password_hash

        if (aluno and aluno["senha_hash"]
                and check_password_hash(aluno["senha_hash"], senha)):
            session.clear()
            session["aluno_id"] = aluno["id"]
            session["aluno_nome"] = aluno["nome"]
            return redirect(url_for("area_aluno"))

        mensagem = "E-mail ou senha incorretos."

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Login - Plataforma EAD</title>
    </head>
    <body style="font-family:Arial;max-width:450px;margin:40px auto;padding:15px">
        <h1>Entrar na plataforma</h1>
        {% if mensagem %}<p style="color:#b91c1c">{{ mensagem }}</p>{% endif %}
        <form method="POST">
            <p><label>E-mail<br>
            <input name="email" type="email" required></label></p>
            <p><label>Senha<br>
            <input name="senha" type="password" required></label></p>
            <button type="submit">Entrar</button>
        </form>
        <p><a href="/esqueci-senha">Esqueci minha senha</a></p>
        <p><a href="/cadastro">Criar uma conta</a></p>
        <p><a href="/">Voltar à página inicial</a></p>
    </body>
    </html>
    """, mensagem=mensagem)


def esqueci_senha():
    mensagem = ""

    if request.method == "POST":
        email = request.form.get("email", "").strip().lower()
        conexao = conectar_banco()

        try:
            aluno = conexao.execute(
                "SELECT id FROM alunos WHERE email = ?", (email,)
            ).fetchone()

            if aluno:
                token = secrets.token_urlsafe(32)
                token_hash = hashlib.sha256(
                    token.encode("utf-8")
                ).hexdigest()
                expiracao = (
                    datetime.utcnow() + timedelta(minutes=30)
                ).isoformat()

                conexao.execute("""
                    INSERT INTO recuperacao_senha
                    (aluno_id, token_hash, expira_em)
                    VALUES (?, ?, ?)
                """, (aluno["id"], token_hash, expiracao))
                conexao.commit()

                # SOMENTE PARA TESTES NO COLAB:
                print(
                    "LINK TEMPORÁRIO DE TESTE:",
                    "/redefinir-senha/" + token
                )

        finally:
            conexao.close()

        mensagem = (
            "Se o e-mail estiver cadastrado, as instruções "
            "de recuperação serão disponibilizadas."
        )

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head><meta charset="UTF-8"><title>Recuperar senha</title></head>
    <body style="font-family:Arial;max-width:450px;margin:40px auto;padding:15px">
        <h1>Esqueci minha senha</h1>
        <p>Informe o e-mail utilizado no cadastro.</p>
        {% if mensagem %}<p>{{ mensagem }}</p>{% endif %}
        <form method="POST">
            <label>E-mail<br>
            <input type="email" name="email" required></label>
            <p><button type="submit">Solicitar recuperação</button></p>
        </form>
        <a href="/login">Voltar ao login</a>
    </body>
    </html>
    """, mensagem=mensagem)


def redefinir_senha(token):
    mensagem = ""
    token_hash = hashlib.sha256(token.encode("utf-8")).hexdigest()
    conexao = conectar_banco()

    try:
        registro = conexao.execute("""
            SELECT id, aluno_id, expira_em
            FROM recuperacao_senha
            WHERE token_hash = ? AND utilizado = 0
        """, (token_hash,)).fetchone()

        valido = False
        if registro:
            expiracao = datetime.fromisoformat(registro["expira_em"])
            valido = datetime.utcnow() < expiracao

        if not valido:
            return (
                "<h1>Link inválido ou expirado</h1>"
                "<p>Solicite uma nova recuperação de senha.</p>"
                "<a href=\"/esqueci-senha\">Recuperar senha</a>",
                400
            )

        if request.method == "POST":
            senha = request.form.get("senha", "")

            if len(senha) < 8:
                mensagem = "A senha precisa ter pelo menos 8 caracteres."
            else:
                conexao.execute("""
                    UPDATE alunos
                    SET senha_hash = ?
                    WHERE id = ?
                """, (
                    generate_password_hash(senha),
                    registro["aluno_id"]
                ))

                conexao.execute("""
                    UPDATE recuperacao_senha
                    SET utilizado = 1
                    WHERE id = ?
                """, (registro["id"],))

                conexao.commit()
                return redirect(url_for("login_aluno"))

    finally:
        conexao.close()

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head><meta charset="UTF-8"><title>Nova senha</title></head>
    <body style="font-family:Arial;max-width:450px;margin:40px auto;padding:15px">
        <h1>Criar nova senha</h1>
        {% if mensagem %}<p style="color:#b91c1c">{{ mensagem }}</p>{% endif %}
        <form method="POST">
            <label>Nova senha (mínimo de 8 caracteres)<br>
            <input type="password" name="senha" minlength="8"
                   autocomplete="new-password" required></label>
            <p><button type="submit">Salvar nova senha</button></p>
        </form>
    </body>
    </html>
    """, mensagem=mensagem)

# Registro dinâmico de segurança das novas rotas para evitar AssertionError
rotas_recuperacao = [
    ("/login", "login_aluno", login_aluno, ["GET", "POST"]),
    ("/esqueci-senha", "esqueci_senha", esqueci_senha, ["GET", "POST"]),
    ("/redefinir-senha/<token>", "redefinir_senha", redefinir_senha, ["GET", "POST"])
]

for caminho, endpoint, funcao, metodos in rotas_recuperacao:
    if endpoint not in app.view_functions:
        app.url_map.add(Rule(caminho, endpoint=endpoint, methods=metodos))
    app.view_functions[endpoint] = funcao

print("Login atualizado e rotas de recuperação de senha configuradas com sucesso!")

Login atualizado e rotas de recuperação de senha configuradas com sucesso!


In [ ]:

import secrets
import hashlib
from datetime import datetime, timedelta
from flask import request, redirect, url_for, render_template_string
from werkzeug.security import generate_password_hash
import sqlite3

# Atualizar a página de login com o link de recuperação
def login_aluno():
    mensagem = ""

    if request.method == "POST":
        email = request.form.get("email", "").strip().lower()
        senha = request.form.get("senha", "")

        conexao = conectar_banco()
        try:
            aluno = conexao.execute("""
                SELECT id, nome, email, senha_hash
                FROM alunos
                WHERE email = ?
            """, (email,)).fetchone()
        finally:
            conexao.close()

        from werkzeug.security import check_password_hash

        if (aluno and aluno["senha_hash"]
                and check_password_hash(aluno["senha_hash"], senha)):
            session.clear()
            session["aluno_id"] = aluno["id"]
            session["aluno_nome"] = aluno["nome"]
            return redirect(url_for("area_aluno"))

        mensagem = "E-mail ou senha incorretos."

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Login - Plataforma EAD</title>
    </head>
    <body style="font-family:Arial;max-width:450px;margin:40px auto;padding:15px">
        <h1>Entrar na plataforma</h1>
        {% if mensagem %}<p style="color:#b91c1c">{{ mensagem }}</p>{% endif %}
        <form method="POST">
            <p><label>E-mail<br>
            <input name="email" type="email" required></label></p>
            <p><label>Senha<br>
            <input name="senha" type="password" required></label></p>
            <button type="submit">Entrar</button>
        </form>
        <p><a href="/esqueci-senha">Esqueci minha senha</a></p>
        <p><a href="/cadastro">Criar uma conta</a></p>
        <p><a href="/">Voltar à página inicial</a></p>
    </body>
    </html>
    """, mensagem=mensagem)


def esqueci_senha():
    mensagem = ""

    if request.method == "POST":
        email = request.form.get("email", "").strip().lower()
        conexao = conectar_banco()

        try:
            aluno = conexao.execute(
                "SELECT id FROM alunos WHERE email = ?", (email,)
            ).fetchone()

            if aluno:
                token = secrets.token_urlsafe(32)
                token_hash = hashlib.sha256(
                    token.encode("utf-8")
                ).hexdigest()
                expiracao = (
                    datetime.utcnow() + timedelta(minutes=30)
                ).isoformat()

                conexao.execute("""
                    INSERT INTO recuperacao_senha
                    (aluno_id, token_hash, expira_em)
                    VALUES (?, ?, ?)
                """, (aluno["id"], token_hash, expiracao))
                conexao.commit()

                # SOMENTE PARA TESTES NO COLAB:
                # não enviamos o token na resposta do navegador.
                print(
                    "LINK TEMPORÁRIO DE TESTE:",
                    "/redefinir-senha/" + token
                )

        finally:
            conexao.close()

        # Mensagem igual exista ou não o e-mail cadastrado.
        mensagem = (
            "Se o e-mail estiver cadastrado, as instruções "
            "de recuperação serão disponibilizadas."
        )

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head><meta charset="UTF-8"><title>Recuperar senha</title></head>
    <body style="font-family:Arial;max-width:450px;margin:40px auto;padding:15px">
        <h1>Esqueci minha senha</h1>
        <p>Informe o e-mail utilizado no cadastro.</p>
        {% if mensagem %}<p>{{ mensagem }}</p>{% endif %}
        <form method="POST">
            <label>E-mail<br>
            <input type="email" name="email" required></label>
            <p><button type="submit">Solicitar recuperação</button></p>
        </form>
        <a href="/login">Voltar ao login</a>
    </body>
    </html>
    """, mensagem=mensagem)


def redefinir_senha(token):
    mensagem = ""
    token_hash = hashlib.sha256(token.encode("utf-8")).hexdigest()
    conexao = conectar_banco()

    try:
        registro = conexao.execute("""
            SELECT id, aluno_id, expira_em
            FROM recuperacao_senha
            WHERE token_hash = ? AND utilizado = 0
        """, (token_hash,)).fetchone()

        valido = False
        if registro:
            expiracao = datetime.fromisoformat(registro["expira_em"])
            valido = datetime.utcnow() < expiracao

        if not valido:
            return (
                "<h1>Link inválido ou expirado</h1>"
                "<p>Solicite uma nova recuperação de senha.</p>"
                "<a href='/esqueci-senha'>Recuperar senha</a>",
                400
            )

        if request.method == "POST":
            senha = request.form.get("senha", "")

            if len(senha) < 8:
                mensagem = "A senha precisa ter pelo menos 8 caracteres."
            else:
                conexao.execute("""
                    UPDATE alunos
                    SET senha_hash = ?
                    WHERE id = ?
                """, (
                    generate_password_hash(senha),
                    registro["aluno_id"]
                ))

                conexao.execute("""
                    UPDATE recuperacao_senha
                    SET utilizado = 1
                    WHERE id = ?
                """, (registro["id"],))

                conexao.commit()
                return redirect(url_for("login_aluno"))

    finally:
        conexao.close()

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head><meta charset="UTF-8"><title>Nova senha</title></head>
    <body style="font-family:Arial;max-width:450px;margin:40px auto;padding:15px">
        <h1>Criar nova senha</h1>
        {% if mensagem %}<p style="color:#b91c1c">{{ mensagem }}</p>{% endif %}
        <form method="POST">
            <label>Nova senha (mínimo de 8 caracteres)<br>
            <input type="password" name="senha" minlength="8"
                   autocomplete="new-password" required></label>
            <p><button type="submit">Salvar nova senha</button></p>
        </form>
    </body>
    </html>
    """, mensagem=mensagem)


# Registrar as rotas ou atualizar suas funções
rotas_recuperacao = [
    ("/login", "login_aluno", login_aluno, ["GET", "POST"]),
    ("/esqueci-senha", "esqueci_senha", esqueci_senha, ["GET", "POST"]),
    ("/redefinir-senha/<token>", "redefinir_senha", redefinir_senha, ["GET", "POST"])
]

for caminho, endpoint, funcao, metodos in rotas_recuperacao:
    if endpoint in app.view_functions:
        app.view_functions[endpoint] = funcao
    else:
        app.add_url_rule(
            caminho,
            endpoint=endpoint,
            view_func=funcao,
            methods=metodos
        )

print("Login atualizado e recuperação de senha configurada!")


Login atualizado e recuperação de senha configurada!


In [ ]:

with app.test_client() as cliente:
    for caminho in ["/login", "/esqueci-senha"]:
        resposta = cliente.get(caminho)
        print(caminho, "->", resposta.status_code)


/login -> 200
/esqueci-senha -> 200


In [ ]:

from flask import session, render_template_string

def pagina_aula(aula_id):
    aluno_id = session["aluno_id"]
    conexao = conectar_banco()

    try:
        aula = conexao.execute("""
            SELECT
                aulas.id,
                aulas.titulo,
                aulas.conteudo,
                aulas.ordem,
                cursos.id AS curso_id,
                cursos.nome AS nome_curso
            FROM aulas
            INNER JOIN cursos ON aulas.curso_id = cursos.id
            WHERE aulas.id = ? AND cursos.ativo = 1
        """, (aula_id,)).fetchone()

        if aula is None:
            return "<h1>Aula não encontrada</h1>", 404

        progresso = conexao.execute("""
            SELECT id
            FROM progresso
            WHERE aluno_id = ? AND aula_id = ?
        """, (aluno_id, aula_id)).fetchone()

    finally:
        conexao.close()

    return render_template_string("""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>{{ aula["titulo"] }} - Plataforma EAD</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                background: #f1f5f9;
                margin: 0;
                padding: 25px 15px;
            }
            main {
                max-width: 800px;
                margin: auto;
                background: white;
                padding: 25px;
                border-radius: 12px;
            }
            h1 { color: #1d4ed8; }
            .botao {
                display: inline-block;
                padding: 12px 18px;
                background: #1d4ed8;
                color: white;
                border: none;
                border-radius: 6px;
                text-decoration: none;
                cursor: pointer;
            }
            .concluida { color: #15803d; font-weight: bold; }
        </style>
    </head>
    <body>
        <main>
            <p><a href="/curso/{{ aula['curso_id'] }}">← Voltar ao curso</a></p>
            <p>Curso: {{ aula["nome_curso"] }}</p>
            <h1>{{ aula["titulo"] }}</h1>
            <p>{{ aula["conteudo"] }}</p>

            {% if progresso %}
                <p class="concluida">✓ Aula concluída!</p>
            {% else %}
                <form method="POST"
                      action="/aula/{{ aula['id'] }}/concluir">
                    <button class="botao" type="submit">
                        Marcar aula como concluída
                    </button>
                </form>
            {% endif %}

            <p><a href="/meu-progresso">Ver meu progresso</a></p>
            <p><a href="/area-aluno">Voltar à área do aluno</a></p>
        </main>
    </body>
    </html>
    """, aula=aula, progresso=progresso)

# Atualizar a função já registrada, mantendo a proteção de login
app.view_functions["pagina_aula"] = exigir_login(pagina_aula)

print("Página de aula atualizada com o botão de conclusão!")


Página de aula atualizada com o botão de conclusão!


In [ ]:
from IPython.display import HTML, display

conexao = conectar_banco()

try:
    aula_teste = conexao.execute("""
        SELECT id, titulo
        FROM aulas
        ORDER BY curso_id, ordem
        LIMIT 1
    """).fetchone()
finally:
    conexao.close()

if aula_teste:
    id_aula = aula_teste["id"]

    with app.test_client() as cliente:
        resposta = cliente.get(f"/aula/{id_aula}")
        html = resposta.get_data(as_text=True)

        print("Status HTTP:", resposta.status_code)
        print("Aula:", aula_teste["titulo"])
        print(
            "Título encontrado:",
            aula_teste["titulo"] in html
        )

        if resposta.status_code == 200:
            display(HTML(html))
else:
    print("Nenhuma aula cadastrada. Cadastre uma aula antes do teste.")

Status HTTP: 302
Aula: Conhecendo o computador
Título encontrado: False


In [ ]:

from flask import abort, redirect, url_for, session

def concluir_aula(aula_id):
    aluno_id = session["aluno_id"]
    conexao = conectar_banco()

    try:
        aula = conexao.execute("""
            SELECT aulas.id
            FROM aulas
            INNER JOIN cursos ON aulas.curso_id = cursos.id
            WHERE aulas.id = ? AND cursos.ativo = 1
        """, (aula_id,)).fetchone()

        if aula is None:
            abort(404)

        conexao.execute("""
            INSERT OR IGNORE INTO progresso (aluno_id, aula_id)
            VALUES (?, ?)
        """, (aluno_id, aula_id))

        conexao.commit()

    finally:
        conexao.close()

    return redirect(url_for("pagina_aula", aula_id=aula_id))

# Substituir a função da rota existente
app.view_functions["concluir_aula"] = exigir_login(concluir_aula)

print("Rota de conclusão atualizada!")


Rota de conclusão atualizada!


In [ ]:

with app.test_client() as cliente:
    resposta = cliente.get(
        "/aula/1",
        follow_redirects=False
    )

    print("Código HTTP:", resposta.status_code)
    print("Redirecionamento:", resposta.headers.get("Location", "-"))


Código HTTP: 302
Redirecionamento: /login


In [ ]:

from flask import render_template_string, session

def area_aluno():
    aluno_id = session["aluno_id"]

    conexao = conectar_banco()

    try:
        cursos = conexao.execute("""
            SELECT
                c.id,
                c.nome,
                COUNT(a.id) AS total_aulas,
                SUM(
                    CASE
                        WHEN p.id IS NOT NULL THEN 1
                        ELSE 0
                    END
                ) AS aulas_concluidas
            FROM cursos c
            LEFT JOIN aulas a
                ON a.curso_id = c.id
            LEFT JOIN progresso p
                ON p.aula_id = a.id
                AND p.aluno_id = ?
            WHERE c.ativo = 1
            GROUP BY c.id, c.nome
            ORDER BY c.nome
        """, (aluno_id,)).fetchall()

    finally:
        conexao.close()

    lista_cursos = []

    for curso in cursos:
        total = curso["total_aulas"] or 0
        concluidas = curso["aulas_concluidas"] or 0

        percentual = (
            round(concluidas / total * 100)
            if total > 0 else 0
        )

        lista_cursos.append({
            "id": curso["id"],
            "nome": curso["nome"],
            "total": total,
            "concluidas": concluidas,
            "percentual": percentual
        })

    return render_template_string("""
        <h1>Minha Área de Estudos</h1>

        <p>Olá, {{ session.get('aluno_nome', 'Aluno') }}!</p>

        <h2>Meus cursos</h2>

        {% for curso in cursos %}
            <hr>
            <h3>{{ curso.nome }}</h3>

            <p>
                Aulas concluídas:
                {{ curso.concluidas }} de {{ curso.total }}
            </p>

            <progress
                value="{{ curso.percentual }}"
                max="100">
            </progress>

            <strong>{{ curso.percentual }}%</strong>

            <p>
                <a href="{{ url_for(
                    'pagina_curso',
                    curso_id=curso.id
                ) }}">
                    Acessar aulas
                </a>
            </p>
        {% else %}
            <p>Nenhum curso disponível no momento.</p>
        {% endfor %}

        <hr>
        <p>
            <a href="{{ url_for('meu_progresso') }}">
                Ver meu progresso geral
            </a>
        </p>

        <p>
            <a href="{{ url_for('sair') }}">Sair da conta</a>
        </p>
    """, cursos=lista_cursos)

app.view_functions["area_aluno"] = exigir_login(area_aluno)

print("Área do aluno atualizada!")


Área do aluno atualizada!


In [ ]:

import traceback

try:
    with app.test_client() as cliente:
        resposta = cliente.get(
            "/area-aluno",
            follow_redirects=True
        )

        print("Código HTTP:", resposta.status_code)
        print("Página retornada:")
        print(resposta.get_data(as_text=True)[:1500])

except Exception:
    traceback.print_exc()


Código HTTP: 200
Página retornada:

    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Login - Plataforma EAD</title>
    </head>
    <body style="font-family:Arial;max-width:450px;margin:40px auto;padding:15px">
        <h1>Entrar na plataforma</h1>
        
        <form method="POST">
            <p><label>E-mail<br>
            <input name="email" type="email" required></label></p>
            <p><label>Senha<br>
            <input name="senha" type="password" required></label></p>
            <button type="submit">Entrar</button>
        </form>
        <p><a href="/esqueci-senha">Esqueci minha senha</a></p>
        <p><a href="/cadastro">Criar uma conta</a></p>
        <p><a href="/">Voltar à página inicial</a></p>
    </body>
    </html>
    


In [2]:

with app.test_client() as cliente:
    resposta = cliente.get("/area-aluno")

    print("Código HTTP:", resposta.status_code)

    if resposta.status_code == 200:
        print("Sucesso! A Área do Aluno abriu corretamente.")
    elif resposta.status_code == 302:
        print("Redirecionamento para:", resposta.headers.get("Location"))
        print("Isso pode ser normal se o aluno não estiver conectado.")
    else:
        print("Ainda existe um erro na página.")
        print(resposta.get_data(as_text=True)[:2000])


NameError: name 'app' is not defined

In [3]:
from sua_aplicacao import app  # Substitua 'sua_aplicacao' pelo nome do seu arquivo/módulo Flask

# Garante que o Flask está rodando no contexto correto para testes
with app.app_context():
    with app.test_client() as cliente:
        resposta = cliente.get("/area-aluno")

        print("Código HTTP:", resposta.status_code)

        if resposta.status_code == 200:
            print("Sucesso! A Área do Aluno abriu corretamente.")
        elif resposta.status_code == 302:
            print("Redirecionamento para:", resposta.headers.get("Location"))
            print("Isso pode ser normal se o aluno não estiver conectado.")
        else:
            print("Ainda existe um erro na página.")
            # Use resposta.get_data(as_text=True) com segurança
            print(resposta.get_data(as_text=True)[:2000])

ModuleNotFoundError: No module named 'sua_aplicacao'